# SAE-Gated RAG: B4 verbalised router (Kaggle)

B4 asks Gemma 3 4B-IT, before answering, to reply with one word: **ANSWER**, **RETRIEVE** or **ABSTAIN**. The prompt is `prompts.VERBALISED_ROUTER`, the decoding is greedy with up to 8 new tokens, and the reply is parsed by `baselines.routers.parse_verbalised`. A reply that can't be parsed falls back to RETRIEVE, and the parse-failure rate is printed for each file. This is the same code as `python -m saerag.baselines.run --routers b4 --backend hf`. B4 does not use labels, so the label rule doesn't matter here. Scoring happens afterwards on your machine, against the `miss_as_retrieve` labels.

## Setup

1. Upload **`saerag-b4-bundle.zip`** as a Kaggle dataset and attach it. It holds the current code, the configs and `data/questions`. The old `saerag-p2-bundle` is missing the B4 code.
2. Add the **`HF_TOKEN`** secret, from an account that accepted the Gemma 3 licence.
3. **Accelerator: GPU T4 x2**, **Internet: On**.
4. **Run All** in a fresh session. The pip cell has to run before torch is imported.

Expect about 30–40 minutes: each file loads the model again (about 1 minute), then generates. If a file fails, run the B4 cell again: finished files are skipped.

## Output

`/kaggle/working/b4_verbalised.zip` contains `runs/routers/b4_verbalised/`:
- `probs_{file}.jsonl`: one-hot router probabilities
- `raw_{file}.jsonl`: the model's raw replies

Unzip it into the repo root, so the files land in `runs/routers/b4_verbalised/`, then score:

```bash
uv run python -m saerag.eval.score --variant miss_as_retrieve \
  --routers b1_always_retrieve b2_never_retrieve b3_oracle b4_verbalised b8_tfidf_lr \
  --files nq_test triviaqa_test popqa_test hotpotqa_test selfaware_test --val-file nq_val
```

In [4]:
FILES = ["nq_val", "triviaqa_val", "nq_test", "triviaqa_test", "popqa_test", "hotpotqa_test", "selfaware_test"]
BATCH_SIZE = 16          # 64 ran out of memory on triviaqa_test (fp32 on T4); lower to 8 if it happens again

import json, os, re, shutil, subprocess, sys
from pathlib import Path

INPUT, WORK, SCRATCH = Path("/kaggle/input"), Path("/kaggle/working"), Path("/tmp/saerag_b4")

found = sorted(INPUT.rglob("BUNDLE_INFO.json"))
found = [p for p in found if "b4" in json.loads(p.read_text()).get("name", "")]
assert len(found) == 1, f"attach saerag-b4-bundle as an input dataset (found {len(found)} copies)"
BUNDLE = found[0].parent
print("bundle:", json.loads(found[0].read_text()))

REPO, DATA = SCRATCH / "repo", SCRATCH / "data"
shutil.rmtree(SCRATCH, ignore_errors=True)
shutil.copytree(BUNDLE / "src", REPO / "src")
shutil.copytree(BUNDLE / "configs", REPO / "configs")
shutil.copytree(BUNDLE / "data", DATA)

# Results go straight to /kaggle/working, so finished files survive a crash and are skipped on a rerun.
RUNS = WORK / "runs"
RUNS.mkdir(parents=True, exist_ok=True)
(REPO / "runs").symlink_to(RUNS, target_is_directory=True)

# The repo targets Python 3.12; drop PEP 695 generics if Kaggle's Python is older.
if sys.version_info < (3, 12):
    schema = REPO / "src/saerag/schema.py"
    s = re.sub(r"def (\w+)\[T\]\(", r"def \1(", schema.read_text())
    s = s.replace("type[T]", "type").replace("-> T:", ":").replace("-> Iterator[T]:", ":")
    schema.write_text(s)

# Batch size for the HF backend.
cfg_path = REPO / "configs/default.yaml"
cfg_text, n_sub = re.subn(r"(\n\s*batch_size:\s*)\d+", rf"\g<1>{BATCH_SIZE}", cfg_path.read_text(), count=1)
assert n_sub == 1, "batch_size not found in configs/default.yaml"
cfg_path.write_text(cfg_text)

missing = [f for f in FILES if not (DATA / "questions" / f"{f}.jsonl").exists()]
assert not missing, f"question files missing from the bundle: {missing}"
print(sys.version)

bundle: {'name': 'saerag-b4-bundle', 'made': '2026-11-10', 'git': '470fabf', 'contents': 'src, configs, data/questions'}
3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]


In [5]:
# Install the team's transformers / accelerate pins WITHOUT touching Kaggle's torch.
# Run this before anything imports torch (fresh session, Run All).
from importlib.metadata import version
if "torch" in sys.modules:
    raise RuntimeError("torch is already loaded. Use Run > Restart & clear cell outputs, then Run All.")
PINS = {"transformers": "5.19.0", "accelerate": "1.15.0"}
TORCH = version("torch")
Path("/tmp/keep_torch.txt").write_text(f"torch=={TORCH}\n")
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-c", "/tmp/keep_torch.txt",
                    *(f"{k}=={v}" for k, v in PINS.items())], capture_output=True, text=True)
print(r.stdout[-1500:], r.stderr[-1500:])
assert r.returncode == 0, "pip failed; read its output above"
check = subprocess.run([sys.executable, "-c",
                        "import torch, transformers, accelerate, sklearn, yaml; "
                        "print(torch.__version__, transformers.__version__, torch.cuda.device_count(), 'GPUs')"],
                       capture_output=True, text=True)
print("check:", check.stdout.strip(), check.stderr[-1500:])
assert check.returncode == 0, "torch + transformers do not import together; see the error above"
assert version("torch") == TORCH, f"torch changed from {TORCH} to {version('torch')}"

 
check: 2.11.0+cu130 5.19.0 2 GPUs 


In [6]:
env = dict(os.environ, SAERAG_DATA=str(DATA), PYTHONPATH=str(REPO / "src"), PYTHONUNBUFFERED="1")
try:
    from kaggle_secrets import UserSecretsClient
    env["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as e:
    print("no HF_TOKEN secret:", e)

env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"   # less fragmentation
B4_DIR = RUNS / "routers/b4_verbalised"

# Run B4 in a child process (same command as on the cluster), one file at a time, and stream
# its log. Files that already have output are skipped, so after a crash just run this cell again.
for f in FILES:
    if (B4_DIR / f"probs_{f}.jsonl").exists():
        print(f"{f}: done already, skipped")
        continue
    cmd = [sys.executable, "-m", "saerag.baselines.run", "--routers", "b4", "--backend", "hf", "--files", f]
    print(" ".join(cmd))
    p = subprocess.Popen(cmd, cwd=REPO, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        if "Loading weights" not in line:
            print(line, end="")
    assert p.wait() == 0, f"B4 failed on {f}; read the log above (out of memory: lower BATCH_SIZE, rerun all)"

/usr/bin/python3 -m saerag.baselines.run --routers b4 --backend hf --files nq_val

loaded google/gemma-3-4b-it as torch.float32 on {'model.vision_tower': 0, 'model.multi_modal_projector': 0, 'model.language_model.embed_tokens': 0, 'lm_head': 0, 'model.language_model.layers.0': 0, 'model.language_model.layers.1': 0, 'model.language_model.layers.2': 0, 'model.language_model.layers.3': 0, 'model.language_model.layers.4': 0, 'model.language_model.layers.5': 0, 'model.language_model.layers.6': 0, 'model.language_model.layers.7': 0, 'model.language_model.layers.8': 0, 'model.language_model.layers.9': 0, 'model.language_model.layers.10': 0, 'model.language_model.layers.11': 0, 'model.language_model.layers.12': 0, 'model.language_model.layers.13': 0, 'model.language_model.layers.14': 0, 'model.language_model.layers.15': 0, 'model.language_model.layers.16': 0, 'model.language_model.layers.17': 0, 'model.language_model.layers.18': 0, 'model.language_model.layers.19': 1, 'model.language_model.lay

In [7]:
import collections
import pandas as pd

OUT = B4_DIR
rows = []
for f in FILES:
    raw = [json.loads(l) for l in open(OUT / f"raw_{f}.jsonl")]
    probs = [json.loads(l) for l in open(OUT / f"probs_{f}.jsonl")]
    picks = collections.Counter(max(r["p"], key=r["p"].get) for r in probs)
    parsed = sum(bool(re.search(r"\b(answer|retrieve|abstain)\b", r["output"], re.I)) for r in raw)
    rows.append({"file": f, "n": len(raw), "parsed": f"{parsed / len(raw):.1%}",
                 **{a: f"{picks[a] / len(probs):.1%}" for a in ("ANSWER", "RETRIEVE", "ABSTAIN")}})
print(pd.DataFrame(rows).to_string(index=False))
print("\nsample replies:", [r["output"] for r in raw[:8]])

zip_base = WORK / "b4_verbalised"
shutil.make_archive(str(zip_base), "zip", root_dir=WORK, base_dir="runs/routers/b4_verbalised")
print("\nwrote", zip_base.with_suffix(".zip"), "- download it and unzip it in the repo root")

          file    n parsed ANSWER RETRIEVE ABSTAIN
        nq_val  500 100.0%  36.6%    62.8%    0.6%
  triviaqa_val  500 100.0%  69.0%    29.4%    1.6%
       nq_test 1000  99.9%  32.1%    67.0%    0.9%
 triviaqa_test 1000 100.0%  66.9%    31.9%    1.2%
    popqa_test 1000 100.0%  13.3%    81.4%    5.3%
 hotpotqa_test 1000 100.0%  24.0%    73.1%    2.9%
selfaware_test  500 100.0%  21.4%    57.6%   21.0%

sample replies: ['RETRIEVE', 'RETRIEVE', 'ANSWER', 'ABSTAIN', 'RETRIEVE', 'ABSTAIN', 'RETRIEVE', 'RETRIEVE']

wrote /kaggle/working/b4_verbalised.zip - download it and unzip it in the repo root
